# 01 — Feedback intelligence system architecture

> **Applied Labs** · **Domain**: PE · **Difficulty**: Advanced · **Reading time**: ~35 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/09_feedback_intelligence/01_architecture.ipynb)

**Prerequisites**:
- [00 — First principles of feedback intelligence](./00_first_principles.ipynb)
- Basic understanding of APIs and data pipelines

**What you will learn**:
- End-to-end architecture for feedback intelligence: ingestion → extraction → trend → alert → action
- Multi-source ingestion and normalisation across reviews, tickets, surveys, and social
- LLM-based aspect extraction with structured JSON output
- Time-series trend detection with configurable smoothing and anomaly thresholds
- Alert and priority systems that route insights to the right teams
- Executive summarisation that turns hundreds of data points into 3-sentence briefs

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "openai>=1.0.0" "sentence-transformers>=2.2.0" "pandas>=2.0.0" "matplotlib>=3.7.0"

import os, json, re, textwrap, hashlib
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Any
from collections import defaultdict, Counter
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

# --- API key (only needed for LLM cells) ---
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY:
    print("OpenAI API key found ✓")
else:
    print("⚠ OPENAI_API_KEY not set — LLM cells will use mock responses")

print("Setup complete ✓")

## Section 1 — System architecture

The feedback intelligence platform processes customer signals through a
**nine-stage pipeline**. Each stage is independently testable and replaceable.

```
┌─────────────┐    ┌──────────────┐    ┌───────────────┐    ┌──────────────────┐
│  Feedback    │───▶│  Ingestion   │───▶│ Deduplication  │───▶│ Aspect Extractor │
│  Sources     │    │  & Normalize │    │  & Merge       │    │  (LLM-based)     │
└─────────────┘    └──────────────┘    └───────────────┘    └──────────┬───────┘
                                                                       │
┌─────────────┐    ┌──────────────┐    ┌───────────────┐    ┌──────────▼───────┐
│   Action     │◀──│  Dashboard   │◀──│  Alert Engine  │◀──│ Sentiment Scorer │
│ Recommender  │    │  Generator   │    │  & Prioritiser │    │ & Trend Detector │
└─────────────┘    └──────────────┘    └───────────────┘    └──────────────────┘
```

**Key design decisions**:
- **Normalise first**: all sources become a common `FeedbackItem` schema
- **Dedup before extraction**: avoid wasting LLM tokens on duplicates
- **Batch extraction**: LLMs process multiple items per call for efficiency
- **Trend detection is streaming-compatible**: uses windowed statistics
- **Alerts are rule-based**: deterministic, auditable, fast

In [ ]:
# --- Pipeline stage interfaces ---

@dataclass
class FeedbackItem:
    """Normalised feedback from any source."""
    id: str
    text: str
    source: str          # review | ticket | survey | social
    timestamp: str       # ISO 8601
    metadata: Dict = field(default_factory=dict)
    dedup_hash: str = ""

@dataclass
class AspectResult:
    """Extracted aspect with sentiment."""
    feedback_id: str
    aspect: str
    opinion: str
    sentiment: str       # positive | negative | neutral
    intensity: float     # 0.0 – 1.0
    evidence: str = ""

@dataclass
class TrendPoint:
    """A single trend observation."""
    aspect: str
    date: str
    count: int
    moving_avg: float
    z_score: float
    is_anomaly: bool

@dataclass
class Alert:
    """System alert for attention."""
    alert_type: str      # volume_spike | sentiment_drop | emerging_aspect
    aspect: str
    severity: str        # critical | warning | info
    message: str
    timestamp: str

# Demonstrate the data flow
sample_flow = [
    ("1. Ingest", "Raw review → FeedbackItem"),
    ("2. Dedup", "FeedbackItem → deduplicated FeedbackItem"),
    ("3. Extract", "FeedbackItem → List[AspectResult]"),
    ("4. Score", "AspectResult → AspectResult with intensity"),
    ("5. Trend", "AspectResults → List[TrendPoint]"),
    ("6. Alert", "TrendPoints → List[Alert]"),
    ("7. Summary", "Alerts + Trends → Executive Summary"),
]
print("Pipeline data flow:\n")
for stage, desc in sample_flow:
    print(f"  {stage:<12} {desc}")
print("\n✓ Architecture defined with typed interfaces")

## Section 2 — Multi-source ingestion

Customer feedback arrives from fundamentally different sources — each with its
own schema, noise profile, and signal density. The ingestion layer normalises
everything to a common `FeedbackItem` schema.

| Source | Raw format | Key challenges |
|---|---|---|
| App Store / Play Store reviews | Star rating + text | Short, informal, multilingual |
| Support tickets (Zendesk, Freshdesk) | Subject + body + metadata | Agent vs customer text, threading |
| Survey responses (NPS, CSAT) | Numeric score + free-text | Low completion, selection bias |
| Social mentions (Twitter/X, Reddit) | Post + thread context | Sarcasm, context-dependent, noisy |

In [ ]:
# --- Source-specific normalisers ---

def normalise_review(raw: Dict) -> FeedbackItem:
    """Normalise app store / product review."""
    text = raw.get("title", "") + ". " + raw.get("body", "")
    return FeedbackItem(
        id=f"review-{raw.get('id', 'unknown')}",
        text=text.strip(),
        source="review",
        timestamp=raw.get("date", datetime.now().isoformat()),
        metadata={"rating": raw.get("rating", 0), "platform": raw.get("platform", "unknown")},
    )

def normalise_ticket(raw: Dict) -> FeedbackItem:
    """Normalise support ticket — extract customer text only."""
    # Filter out agent responses
    messages = raw.get("messages", [])
    customer_msgs = [m["text"] for m in messages if m.get("role") == "customer"]
    text = " ".join(customer_msgs) if customer_msgs else raw.get("subject", "")
    return FeedbackItem(
        id=f"ticket-{raw.get('id', 'unknown')}",
        text=text.strip(),
        source="ticket",
        timestamp=raw.get("created_at", datetime.now().isoformat()),
        metadata={"priority": raw.get("priority", "normal"), "category": raw.get("category", "")},
    )

def normalise_survey(raw: Dict) -> FeedbackItem:
    """Normalise survey response."""
    text = raw.get("free_text", "")
    return FeedbackItem(
        id=f"survey-{raw.get('id', 'unknown')}",
        text=text.strip(),
        source="survey",
        timestamp=raw.get("submitted_at", datetime.now().isoformat()),
        metadata={"nps_score": raw.get("nps_score", 0), "survey_type": raw.get("type", "nps")},
    )

def normalise_social(raw: Dict) -> FeedbackItem:
    """Normalise social media mention."""
    text = raw.get("text", "")
    # Remove @mentions and URLs for cleaner extraction
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"https?://\S+", "", text)
    return FeedbackItem(
        id=f"social-{raw.get('id', 'unknown')}",
        text=text.strip(),
        source="social",
        timestamp=raw.get("posted_at", datetime.now().isoformat()),
        metadata={"platform": raw.get("platform", ""), "engagement": raw.get("likes", 0)},
    )

NORMALISERS = {
    "review": normalise_review,
    "ticket": normalise_ticket,
    "survey": normalise_survey,
    "social": normalise_social,
}

# --- Demo: ingest from all 4 sources ---
raw_samples = [
    {"source": "review", "id": "r1", "title": "Love the app", "body": "But login is broken since last update", "rating": 2, "date": "2025-01-20", "platform": "App Store"},
    {"source": "ticket", "id": "t1", "subject": "Cannot login", "messages": [{"role": "customer", "text": "I keep getting timeout errors when trying to log in. This started 2 days ago."}, {"role": "agent", "text": "We're looking into this."}], "created_at": "2025-01-21", "priority": "high"},
    {"source": "survey", "id": "s1", "nps_score": 3, "free_text": "Would be 10/10 if billing wasn't so confusing. Had duplicate charges twice.", "submitted_at": "2025-01-19", "type": "nps"},
    {"source": "social", "id": "x1", "text": "@acmeapp your new dashboard is gorgeous but WHERE is the export button?? https://t.co/abc123", "posted_at": "2025-01-22", "platform": "twitter", "likes": 47},
]

print("Multi-source ingestion demo:\n")
normalised: List[FeedbackItem] = []
for raw in raw_samples:
    src = raw.pop("source")
    item = NORMALISERS[src](raw)
    normalised.append(item)
    print(f"  [{item.source:>7}] {item.id}: "{item.text[:70]}..."")
    print(f"           metadata: {item.metadata}")
    print()

print(f"✓ Ingested {len(normalised)} items from 4 different sources into common schema")

## Section 3 — Aspect extraction engine

The extraction engine uses LLM-based prompting to produce structured JSON
output from raw feedback text. The prompt is carefully designed to:
- Output a **consistent JSON schema**
- Use a **fixed aspect taxonomy** (prevents aspect proliferation)
- Include **verbatim evidence** for auditability
- Handle **multi-aspect** reviews correctly

We also implement a **deduplication** layer that runs before extraction
to avoid wasting LLM tokens on near-duplicate feedback.

In [ ]:
def compute_dedup_hash(text: str) -> str:
    """Content-based deduplication hash (normalised lowercase, stripped)."""
    normalised = re.sub(r"\s+", " ", text.lower().strip())
    return hashlib.md5(normalised.encode()).hexdigest()

def deduplicate(items: List[FeedbackItem], similarity_threshold: float = 1.0) -> List[FeedbackItem]:
    """Remove exact duplicates by content hash. Returns deduplicated list."""
    seen_hashes: set = set()
    unique: List[FeedbackItem] = []
    duplicates = 0
    for item in items:
        h = compute_dedup_hash(item.text)
        item.dedup_hash = h
        if h not in seen_hashes:
            seen_hashes.add(h)
            unique.append(item)
        else:
            duplicates += 1
    if duplicates:
        print(f"  Dedup: removed {duplicates} duplicate(s)")
    return unique

# --- LLM extraction prompt ---
ASPECT_EXTRACTION_PROMPT = """You are a feedback intelligence system. Extract all aspect-level opinions.

ASPECT CATEGORIES (use exactly these):
- product_quality, customer_support, pricing, ui_ux, performance
- login, onboarding, documentation, reliability, billing, general

For each aspect mentioned, extract:
- aspect: category from above list
- opinion: what the customer said about it (paraphrase)
- sentiment: positive | negative | neutral
- intensity: 0.0 to 1.0 (0.1=mild, 0.5=moderate, 0.9=extreme)
- evidence: verbatim quote from the text

Return ONLY a JSON array. If no feedback found, return [].

FEEDBACK: "{text}"
"""

def mock_llm_extract(text: str) -> List[Dict]:
    """Simulate LLM extraction for demonstration (no API call)."""
    results = []
    text_lower = text.lower()
    aspect_signals = {
        "login": ["login", "log in", "sign in", "timeout", "authentication"],
        "billing": ["billing", "charge", "invoice", "payment", "refund", "duplicate charge"],
        "ui_ux": ["dashboard", "ui", "ux", "interface", "design", "button", "navigation", "export"],
        "customer_support": ["support", "help", "agent", "response time", "waiting"],
        "performance": ["slow", "fast", "crash", "lag", "speed", "load"],
        "pricing": ["price", "expensive", "cheap", "cost", "pricing"],
        "product_quality": ["quality", "product", "love", "great", "amazing"],
        "onboarding": ["setup", "onboard", "getting started", "install"],
        "reliability": ["reliable", "uptime", "downtime", "outage", "broken"],
    }
    positive_kw = {"love", "great", "excellent", "amazing", "gorgeous", "beautiful", "good", "breeze"}
    negative_kw = {"broken", "confusing", "slow", "timeout", "error", "bad", "terrible", "missing"}

    for aspect, keywords in aspect_signals.items():
        if any(kw in text_lower for kw in keywords):
            words = set(text_lower.split())
            pos = words & positive_kw
            neg = words & negative_kw
            sent = "positive" if pos and not neg else ("negative" if neg else "neutral")
            intensity = 0.7 if (pos or neg) else 0.4
            results.append({
                "aspect": aspect,
                "opinion": f"Customer mentioned {aspect}",
                "sentiment": sent,
                "intensity": intensity,
                "evidence": text[:80],
            })
    if not results:
        results.append({"aspect": "general", "opinion": "General feedback",
                        "sentiment": "neutral", "intensity": 0.3, "evidence": text[:80]})
    return results

# --- Demo on our normalised items ---
print("Aspect extraction (mock LLM):\n")
all_aspects: List[AspectResult] = []
deduped = deduplicate(normalised)

for item in deduped:
    extractions = mock_llm_extract(item.text)
    print(f"  {item.id}:")
    for ext in extractions:
        ar = AspectResult(
            feedback_id=item.id, aspect=ext["aspect"], opinion=ext["opinion"],
            sentiment=ext["sentiment"], intensity=ext["intensity"], evidence=ext["evidence"]
        )
        all_aspects.append(ar)
        print(f"    [{ar.sentiment:>8} {ar.intensity:.1f}] {ar.aspect}: {ar.opinion}")
    print()

print(f"✓ Extracted {len(all_aspects)} aspect-opinion pairs from {len(deduped)} feedback items")

## Section 4 — Trend detection pipeline

The trend detector aggregates aspect extraction results into **daily counts**,
applies **exponential moving average (EMA)** smoothing, and flags anomalies
using **z-score thresholds**.

The pipeline supports configurable parameters:
- `window_size`: smoothing window (default: 7 days)
- `z_threshold`: anomaly sensitivity (default: 2.0)
- `min_observations`: minimum days before alerting (avoids false positives on sparse data)

In [ ]:
class TrendDetector:
    """Configurable trend detection over aspect time series."""

    def __init__(self, window: int = 7, z_threshold: float = 2.0, min_obs: int = 5):
        self.window = window
        self.z_threshold = z_threshold
        self.min_obs = min_obs

    def analyse(self, df: pd.DataFrame) -> Dict[str, List[TrendPoint]]:
        """Analyse a DataFrame with columns = aspects, index = dates."""
        results: Dict[str, List[TrendPoint]] = {}
        for aspect in df.columns:
            series = df[aspect].astype(float)
            rolling_mean = series.rolling(window=self.window, min_periods=3).mean()
            rolling_std = series.rolling(window=self.window, min_periods=3).std().replace(0, 1)
            z_scores = (series - rolling_mean) / rolling_std

            points = []
            for i, date in enumerate(df.index):
                points.append(TrendPoint(
                    aspect=aspect,
                    date=date.strftime("%Y-%m-%d") if hasattr(date, "strftime") else str(date),
                    count=int(series.iloc[i]),
                    moving_avg=round(rolling_mean.iloc[i], 2) if not pd.isna(rolling_mean.iloc[i]) else 0,
                    z_score=round(z_scores.iloc[i], 2) if not pd.isna(z_scores.iloc[i]) else 0,
                    is_anomaly=bool(z_scores.iloc[i] > self.z_threshold) if not pd.isna(z_scores.iloc[i]) else False,
                ))
            results[aspect] = points
        return results

# --- Generate sample 30-day data for demo ---
np.random.seed(42)
days = 30
date_range = pd.date_range("2025-01-01", periods=days, freq="D")
aspects_data = {
    "login": np.random.poisson(4, days).astype(float),
    "billing": np.random.poisson(3, days).astype(float),
    "ui_ux": np.random.poisson(5, days).astype(float),
    "performance": np.random.poisson(4, days).astype(float),
}
# Inject anomalies
aspects_data["login"][15:] += np.linspace(2, 15, days - 15)
aspects_data["billing"][27:] += np.array([8, 10, 6])[:days - 27]

df_demo = pd.DataFrame(aspects_data, index=date_range)
detector = TrendDetector(window=7, z_threshold=2.0)
trend_results = detector.analyse(df_demo)

# Visualise
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for ax, aspect in zip(axes.flat, trend_results.keys()):
    points = trend_results[aspect]
    dates = [p.date for p in points]
    counts = [p.count for p in points]
    ma = [p.moving_avg for p in points]
    anomalies = [i for i, p in enumerate(points) if p.is_anomaly]

    ax.plot(range(len(dates)), counts, alpha=0.6, label="daily count")
    ax.plot(range(len(dates)), ma, color="black", linestyle="--", label="7-day MA")
    if anomalies:
        ax.scatter(anomalies, [counts[i] for i in anomalies],
                   color="red", s=60, zorder=5, marker="^", label="anomaly")
    ax.set_title(aspect)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle("Trend Detection — 4 Aspects over 30 Days", fontsize=13)
plt.tight_layout()
plt.show()

# Summary
print("\nAnomaly summary:")
for aspect, points in trend_results.items():
    anom = [p for p in points if p.is_anomaly]
    if anom:
        print(f"  {aspect}: {len(anom)} anomalous days, first: {anom[0].date}")
    else:
        print(f"  {aspect}: no anomalies detected")
print("\n✓ Trend detector with configurable window and z-score threshold")

## Section 5 — Alert and priority system

The alert engine converts trend anomalies into **actionable alerts** using
deterministic rules. Alerts are categorised by type and severity:

| Alert type | Trigger | Severity |
|---|---|---|
| `volume_spike` | Z-score > threshold for an aspect | critical if z > 3, warning if z > 2 |
| `sentiment_drop` | Average sentiment drops below threshold for N days | critical if < -0.5 |
| `emerging_aspect` | New aspect appears that wasn't seen in prior window | info |

Each alert gets a **priority score** for ranking.

In [ ]:
class AlertEngine:
    """Rule-based alert engine for feedback intelligence."""

    def __init__(self, z_critical: float = 3.0, z_warning: float = 2.0):
        self.z_critical = z_critical
        self.z_warning = z_warning

    def generate_alerts(self, trend_results: Dict[str, List[TrendPoint]]) -> List[Alert]:
        """Generate alerts from trend analysis results."""
        alerts: List[Alert] = []

        for aspect, points in trend_results.items():
            # --- Volume spike alerts ---
            for p in points:
                if p.is_anomaly:
                    severity = "critical" if p.z_score > self.z_critical else "warning"
                    alerts.append(Alert(
                        alert_type="volume_spike",
                        aspect=aspect,
                        severity=severity,
                        message=f"{aspect} mentions spiked to {p.count} "
                                f"(z={p.z_score:.1f}, MA={p.moving_avg:.1f})",
                        timestamp=p.date,
                    ))

            # --- Sustained increase detection ---
            recent = points[-7:] if len(points) >= 7 else points
            if len(recent) >= 5:
                counts = [p.count for p in recent]
                slope = (counts[-1] - counts[0]) / len(counts)
                if slope > 2.0:
                    alerts.append(Alert(
                        alert_type="volume_spike",
                        aspect=aspect,
                        severity="warning",
                        message=f"{aspect} shows sustained increase "
                                f"(slope={slope:.1f}/day over last {len(recent)} days)",
                        timestamp=recent[-1].date,
                    ))

        # Sort by severity (critical first)
        severity_order = {"critical": 0, "warning": 1, "info": 2}
        alerts.sort(key=lambda a: (severity_order.get(a.severity, 9), a.timestamp))
        return alerts

    def format_alert_feed(self, alerts: List[Alert]) -> str:
        """Format alerts as a human-readable feed."""
        lines = []
        icons = {"critical": "🔴", "warning": "🟡", "info": "🔵"}
        for a in alerts:
            icon = icons.get(a.severity, "⚪")
            lines.append(f"  {icon} [{a.severity:>8}] {a.timestamp} | {a.alert_type}: {a.message}")
        return "\n".join(lines)

# --- Generate alerts from our trend data ---
engine = AlertEngine(z_critical=3.0, z_warning=2.0)
alerts = engine.generate_alerts(trend_results)

print(f"Alert feed ({len(alerts)} alerts):\n")
print(engine.format_alert_feed(alerts))

# --- Priority scoring ---
def alert_priority(alert: Alert) -> float:
    """Score alert priority for ranking."""
    base = {"critical": 100, "warning": 50, "info": 10}.get(alert.severity, 0)
    type_weight = {"volume_spike": 1.5, "sentiment_drop": 2.0, "emerging_aspect": 1.0}.get(alert.alert_type, 1.0)
    return base * type_weight

print(f"\nTop 5 alerts by priority:")
top_alerts = sorted(alerts, key=alert_priority, reverse=True)[:5]
for a in top_alerts:
    print(f"  [P{alert_priority(a):>5.0f}] {a.message[:80]}")
print(f"\n✓ Alert engine generated {len(alerts)} alerts from trend analysis")

## Section 6 — Insight summarisation

The final pipeline stage aggregates all aspect data, trend signals, and alerts
into a concise **executive summary** suitable for product managers and
leadership.

The summariser groups aspects into themes and produces a brief like:
> *"Top 3 issues this week: (1) login failures up 200%, affecting 45 users;
> (2) billing confusion steady at 8/day; (3) new demand for CSV export feature."*

We design the LLM prompt and build a fallback template-based summariser.

In [ ]:
SUMMARY_PROMPT = """You are a feedback intelligence analyst. Generate a concise executive
summary from the following feedback data for the period {start_date} to {end_date}.

ASPECT SUMMARY:
{aspect_summary}

ALERTS:
{alert_summary}

TREND DATA:
{trend_summary}

Write a 3–5 sentence executive summary covering:
1. Top issues by volume and severity (with numbers)
2. Emerging trends (what's getting worse/better)
3. Recommended actions (specific, prioritised)

Use professional tone. Include specific numbers. Be actionable.
"""

def template_summariser(
    aspect_counts: Dict[str, int],
    alerts: List[Alert],
    trend_slopes: Dict[str, float],
    start_date: str = "2025-01-01",
    end_date: str = "2025-01-30",
) -> str:
    """Template-based executive summary (no LLM needed)."""
    # Rank aspects by volume
    ranked = sorted(aspect_counts.items(), key=lambda x: x[1], reverse=True)
    top_3 = ranked[:3]

    # Identify worsening trends
    worsening = [(a, s) for a, s in trend_slopes.items() if s > 0.5]
    worsening.sort(key=lambda x: x[1], reverse=True)

    # Count critical alerts
    critical = [a for a in alerts if a.severity == "critical"]

    lines = [
        f"📊 Executive Summary — {start_date} to {end_date}",
        f"{'=' * 55}",
        "",
        f"Top issues by volume:",
    ]
    for i, (aspect, count) in enumerate(top_3, 1):
        trend = trend_slopes.get(aspect, 0)
        arrow = "↑" if trend > 0.5 else ("→" if trend > -0.5 else "↓")
        lines.append(f"  {i}. {aspect}: {count} mentions ({arrow} {'increasing' if trend > 0.5 else 'stable'})")

    if worsening:
        lines.append(f"\n⚠ Worsening trends:")
        for asp, slope in worsening[:3]:
            lines.append(f"  - {asp}: +{slope:.1f} mentions/day trend")

    if critical:
        lines.append(f"\n🔴 {len(critical)} critical alert(s):")
        for a in critical[:3]:
            lines.append(f"  - {a.message[:70]}")

    lines.append(f"\nRecommended actions:")
    if worsening:
        lines.append(f"  1. Investigate {worsening[0][0]} — fastest-growing issue")
    if top_3:
        lines.append(f"  2. Address {top_3[0][0]} — highest volume ({top_3[0][1]} mentions)")
    lines.append(f"  3. Review all {len(critical)} critical alerts for immediate resolution")

    return "\n".join(lines)

# --- Build summary from our demo data ---
aspect_counts = {asp: int(df_demo[asp].sum()) for asp in df_demo.columns}
trend_slopes = {}
for asp in df_demo.columns:
    vals = df_demo[asp].values
    slope = (vals[-7:].mean() - vals[:7].mean()) / 7
    trend_slopes[asp] = round(slope, 2)

summary = template_summariser(aspect_counts, alerts, trend_slopes)
print(summary)
print("\n✓ Executive summary generated from pipeline outputs")

In [ ]:
# --- Architecture validation ---
pipeline_checks = {
    "ingestion": len(normalised) == 4,
    "deduplication": len(deduped) <= len(normalised),
    "extraction": len(all_aspects) > 0,
    "trend_detection": len(trend_results) > 0,
    "alerting": len(alerts) >= 0,
    "summarisation": len(summary) > 50,
}
for stage, passed in pipeline_checks.items():
    status = "✓" if passed else "✗"
    print(f"  {status} {stage}")
print(f"\n✓ {sum(pipeline_checks.values())}/{len(pipeline_checks)} pipeline stages verified")

## Exercises

1. **Add a new source normaliser** — Write a `normalise_intercom()` function
   that handles Intercom conversation exports (JSON with `conversation_parts`).
   Handle both user and admin messages, extracting only user text.

2. **Implement near-duplicate detection** — Extend `deduplicate()` to catch
   semantically similar (not just identical) feedback using cosine similarity on
   TF-IDF vectors. Set a threshold of 0.85 and test on 10 pairs of similar reviews.

3. **Build a real LLM extractor** — Replace `mock_llm_extract()` with a real
   OpenAI API call using the `ASPECT_EXTRACTION_PROMPT`. Parse the JSON response
   and handle error cases (malformed JSON, refusals, rate limits).

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | Normalise all feedback sources to a common schema before processing |
| 2 | Deduplicate before LLM extraction to save tokens and avoid double-counting |
| 3 | LLM-based extraction with fixed taxonomy + JSON output = structured, consistent results |
| 4 | Trend detection uses windowed z-scores — configurable, streaming-compatible, interpretable |
| 5 | Alerts are rule-based: deterministic, auditable, and fast to evaluate |
| 6 | Executive summaries convert hundreds of data points into 3–5 actionable sentences |
| 7 | Each pipeline stage is independently testable and replaceable |

## What's Next

In **02 — Build**, we construct the full platform: generate a synthetic
200-item dataset, run end-to-end extraction, build trend analysis with
visualisations, and generate an executive summary.

---

## References

1. Liu, B., *Sentiment Analysis and Opinion Mining*, Morgan & Claypool, 2012 — <https://www.cs.uic.edu/~liub/FBS/SentimentAnalysis-and-OpinionMining.pdf>
2. Pontiki, M. et al., *SemEval-2016 Task 5: Aspect-Based Sentiment Analysis*, 2016 — <https://aclanthology.org/S16-1002/>
3. OpenAI, *Structured Outputs Guide*, 2024 — <https://platform.openai.com/docs/guides/structured-outputs>
4. Medallia Experience Cloud — <https://www.medallia.com/>